
<a href="https://colab.research.google.com/github/adenikeadewumi/python-ml-WIEOAU/blob/main/00_careers_in_ai_ml/python_in_ai_ml.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Python in AI/ML — Interactive Overview

This notebook gives you a hands-on feel for how Python is used at each stage of the AI/ML pipeline. Read the README.md in this folder for the full career guide.

---

## Stage 1: Data Collection

**In the real world**, data comes from databases, APIs, web scraping, file systems, and sensors. The first job of any ML project is gathering and organising raw data.

Python is the tool of choice because its ecosystem covers every data source imaginable.

In [ ]:
# Simulating data collection — in practice you'd use requests, sqlalchemy, etc.
import pandas as pd
import numpy as np

np.random.seed(42)

# Imagine this data came from a database or API
raw_data = {
    "user_id":     range(1, 101),
    "age":         np.random.randint(18, 65, 100),
    "income":      np.random.normal(50000, 15000, 100).round(0),
    "tenure_days": np.random.exponential(365, 100).round(0),
    "clicked_ad":  np.random.choice([0, 1], 100, p=[0.7, 0.3]),
    "region":      np.random.choice(["North", "South", "East", "West"], 100),
}

df = pd.DataFrame(raw_data)
print(f"Collected {len(df)} records")
print(df.head())

## Stage 2: Exploratory Data Analysis (EDA)

**Before building any model**, data scientists explore the data to understand its structure, spot problems, and generate hypotheses.

EDA answers questions like: What is the distribution of my target variable? Are there outliers? Are features correlated? How much data is missing?

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Basic EDA
print("Dataset shape:", df.shape)
print("
Data types:")
print(df.dtypes)
print("
Missing values:", df.isnull().sum().sum())
print("
Target variable distribution:")
print(df["clicked_ad"].value_counts())
print(f"Click rate: {df['clicked_ad'].mean():.1%}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Age distribution
axes[0].hist(df["age"], bins=20, color="steelblue", edgecolor="white")
axes[0].set_title("Age Distribution")
axes[0].set_xlabel("Age")

# Income by click outcome
df.boxplot(column="income", by="clicked_ad", ax=axes[1])
axes[1].set_title("Income by Click Outcome")
axes[1].set_xlabel("Clicked Ad (0=No, 1=Yes)")

# Click rate by region
region_rates = df.groupby("region")["clicked_ad"].mean().sort_values()
region_rates.plot(kind="barh", ax=axes[2], color="coral")
axes[2].set_title("Click Rate by Region")
axes[2].set_xlabel("Click Rate")

plt.tight_layout()
plt.show()

## Stage 3: Feature Engineering

**Raw data is rarely in the right form for a model.** Feature engineering transforms raw variables into representations that help the model learn patterns.

This is where domain knowledge matters most — a healthcare data scientist knows that "age squared" or "years since diagnosis" might matter; the algorithm alone does not.

In [ ]:
# Feature engineering
df_features = df.copy()

# Bin continuous age into categories
df_features["age_group"] = pd.cut(
    df["age"],
    bins=[0, 25, 35, 45, 65],
    labels=["18-25", "26-35", "36-45", "46-65"]
)

# Log-transform skewed income
df_features["log_income"] = np.log1p(df["income"].clip(lower=0))

# Convert tenure from days to months
df_features["tenure_months"] = df["tenure_days"] / 30

# One-hot encode the region column
df_encoded = pd.get_dummies(df_features, columns=["region", "age_group"], drop_first=True)

print("Features after engineering:")
print(df_encoded.columns.tolist())
print(f"
Shape: {df_encoded.shape}")

## Stage 4: Model Training and Evaluation

**Now we build the actual model.** In practice, data scientists try several algorithms and compare them. The best model is chosen based on cross-validated performance metrics, not just training accuracy.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler

# Prepare features and target
feature_cols = [c for c in df_encoded.columns if c not in ["user_id", "clicked_ad"]]
X = df_encoded[feature_cols].select_dtypes(include=[np.number])
y = df_encoded["clicked_ad"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Try multiple models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

print("Model comparison (5-fold CV ROC-AUC):")
print("-" * 45)
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring="roc_auc")
    print(f"  {name:<25} {scores.mean():.3f} +/- {scores.std():.3f}")

## Stage 5: Production Deployment

**A model that lives only in a Jupyter notebook has zero business value.** Deployment means making your model accessible to users or other systems, usually via an API.

This is where ML Engineers come in — they take the notebook code and build it into a reliable, scalable service.

In [ ]:
# This is what a model serving API looks like
# In production you would run this with: uvicorn app:app --host 0.0.0.0

api_code = '''
from fastapi import FastAPI
from pydantic import BaseModel
import joblib

app = FastAPI(title="Ad Click Predictor")

# Load the trained model (saved with joblib.dump(model, "model.pkl"))
model = joblib.load("model.pkl")

class UserFeatures(BaseModel):
    age: int
    income: float
    tenure_days: float
    region: str

@app.post("/predict")
def predict(features: UserFeatures):
    # Transform input (same steps as training!)
    # ... feature engineering ...
    
    prediction = model.predict_proba([feature_vector])[0]
    return {
        "will_click": bool(prediction[1] > 0.5),
        "probability": round(float(prediction[1]), 3)
    }
'''

print("Example FastAPI deployment code:")
print(api_code)
print()
print("The model would be called via HTTP:")
print('  POST /predict')
print('  Body: {"age": 28, "income": 62000, "tenure_days": 450, "region": "North"}')
print('  Response: {"will_click": true, "probability": 0.71}')

---

## The ML Engineer's Checklist for Production

Before any model goes live, a good ML engineer asks:

1. **Reproducibility** — Can the training pipeline be re-run from scratch and produce the same result?
2. **Data validation** — What happens if the input data has missing values or unexpected types?
3. **Monitoring** — How will we know if the model starts performing poorly in production?
4. **Data drift** — The world changes. Income distributions in 2024 are different from 2020. When do we retrain?
5. **Latency** — Does the model respond fast enough for the use case? (Real-time fraud detection needs <50ms)
6. **Fallback** — What happens if the model service goes down?
7. **Fairness** — Is the model performing equally well for all demographic groups?

These are the questions that distinguish a toy model from a production system.

---

## Further Reading

See [README.md](README.md) for the complete career guide, salary data, and learning roadmaps.
